# ESMC Debugging Notebook


In [12]:
import torch
from esm.models.esmc import ESMC
from transformer_lens import HookedESMC, SupportedESMCConfig
from esm.tokenization import get_esmc_model_tokenizers
from esm.pretrained import ESMC_600M_202412, ESMC_300M_202412

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "esmc_300m"
sequence = "MMPISAAHKIMQTSDETYTTVGNIKVKCEEVARSTLQIPGSVNALSEKCEEVARSSLPIPGSVNALSENLLWCWLEGSKGLRSSNKIKGEIYGRRTAGIDIGLANADKVHKAPKAMRIRGASREFHSVCVMPQ"

config = SupportedESMCConfig(
    use_attn_result=False,
    use_split_qkv_input=False,
    use_hook_mlp_in=False,
    use_attn_in=False,
    esmc_use_torch_layer_norm=True,
    esmc_use_torch_attention_calc=True,
    esmc_use_org_rotary=True,
    esmc_capture_activations_before_normalization=False
)

print(f"Device: {device}, Model: {model_name}")


Device: cuda, Model: esmc_300m


In [13]:
# Tokenize
tokenizer = get_esmc_model_tokenizers()
tokenizer_res = tokenizer([sequence], return_tensors="pt", padding=True)
sequence_tokens = tokenizer_res['input_ids'].to(device)  # type: ignore[attr-defined]
sequence_id = tokenizer_res['attention_mask'].to(device)  # type: ignore[attr-define`d]
print(f"Sequence tokens shape: {sequence_tokens.shape}")


Sequence tokens shape: torch.Size([1, 135])


In [14]:
# Load both models
device_obj = torch.device(device)
esmc_original = ESMC_600M_202412(device=device) if model_name == "esmc_600m" else ESMC_300M_202412(device=device)
esmc_original = esmc_original.to(device).to(torch.float32).eval()
esmc_hooked = HookedESMC.from_pretrained(esmc_cfg=config, model_name=model_name, device=device).eval()
print("Both models loaded")


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

/home/galkesten/miniconda3/envs/transformer_lens_cuda12_4/lib/python3.10/site-packages/esm/pretrained.py:70: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch

Moving model to device:  cuda
Loaded pretrained model esmc_300m into HookedESMC
Both models loaded


In [15]:
# Register hooks for first block
hooks = {'orig_attn': {}, 'orig_ffn': {}, 'orig_ln_qkv': {}, 
         'orig_rotary_q_in': {}, 'orig_rotary_k_in': {}, 'orig_rotary_q_out': {}, 'orig_rotary_k_out': {},
         'orig_q_ln_in': {}, 'orig_q_ln_out': {}, 'orig_k_ln_in': {}, 'orig_k_ln_out': {},
         'hook_attn': {}, 'hook_mlp': {}, 'hook_ln1': {},
         'hook_rotary_q_in': {}, 'hook_rotary_k_in': {}, 'hook_rotary_q_out': {}, 'hook_rotary_k_out': {},
         'hook_q_ln_in': {}, 'hook_q_ln_out': {}, 'hook_k_ln_in': {}, 'hook_k_ln_out': {}}

def make_hook(key): return lambda m, i, o: hooks[key].update({'output': o.detach().clone()})

def make_ln_hook(key_in, key_out):
    def hook(m, i, o):
        # Capture input - layer norm takes a single tensor
        if isinstance(i, tuple):
            hooks[key_in].update({'output': i[0].detach().clone()})
        else:
            hooks[key_in].update({'output': i.detach().clone()})
        # Capture output
        hooks[key_out].update({'output': o.detach().clone()})
    return hook

def make_rotary_hook(key_q_in, key_k_in, key_q_out, key_k_out): 
    def hook(m, i, o):
        # Capture input - rotary takes (q, k) as input tuple
        if isinstance(i, tuple) and len(i) >= 2:
            hooks[key_q_in].update({'output': i[0].detach().clone()})  # q
            hooks[key_k_in].update({'output': i[1].detach().clone()})  # k
        # Capture output - rotary returns (q, k) as output tuple
        if isinstance(o, tuple) and len(o) >= 2:
            hooks[key_q_out].update({'output': o[0].detach().clone()})  # q
            hooks[key_k_out].update({'output': o[1].detach().clone()})  # k
    return hook

h1 = esmc_original.transformer.blocks[0].attn.out_proj.register_forward_hook(make_hook('orig_attn'))
h2 = esmc_original.transformer.blocks[0].ffn.register_forward_hook(make_hook('orig_ffn'))
h3 = esmc_original.transformer.blocks[0].attn.layernorm_qkv[0].register_forward_hook(make_hook('orig_ln_qkv'))
h4 = esmc_original.transformer.blocks[0].attn.rotary.register_forward_hook(
    make_rotary_hook('orig_rotary_q_in', 'orig_rotary_k_in', 'orig_rotary_q_out', 'orig_rotary_k_out'))
h5 = esmc_original.transformer.blocks[0].attn.q_ln.register_forward_hook(
    make_ln_hook('orig_q_ln_in', 'orig_q_ln_out'))
h6_orig = esmc_original.transformer.blocks[0].attn.k_ln.register_forward_hook(
    make_ln_hook('orig_k_ln_in', 'orig_k_ln_out'))
h7 = esmc_hooked.blocks[0].attn.register_forward_hook(make_hook('hook_attn'))
h8 = esmc_hooked.blocks[0].mlp.register_forward_hook(make_hook('hook_mlp'))
h9 = esmc_hooked.blocks[0].ln1.register_forward_hook(make_hook('hook_ln1'))
h10 = esmc_hooked.blocks[0].attn.rotary.register_forward_hook(
    make_rotary_hook('hook_rotary_q_in', 'hook_rotary_k_in', 'hook_rotary_q_out', 'hook_rotary_k_out'))
h11 = esmc_hooked.blocks[0].attn.q_ln.register_forward_hook(
    make_ln_hook('hook_q_ln_in', 'hook_q_ln_out'))
h12 = esmc_hooked.blocks[0].attn.k_ln.register_forward_hook(
    make_ln_hook('hook_k_ln_in', 'hook_k_ln_out'))
print("Hooks registered")


Hooks registered


In [16]:
# Forward pass
with torch.no_grad():
    out1 = esmc_original.forward(sequence_tokens=sequence_tokens, sequence_id=sequence_id)
    out2 = esmc_hooked.forward(sequence_tokens=sequence_tokens, sequence_id=sequence_id, return_type="logits")
print("Forward pass completed")
print(f"q_ln: orig_in={'output' in hooks['orig_q_ln_in']}, orig_out={'output' in hooks['orig_q_ln_out']}, hook_in={'output' in hooks['hook_q_ln_in']}, hook_out={'output' in hooks['hook_q_ln_out']}")
print(f"k_ln: orig_in={'output' in hooks['orig_k_ln_in']}, orig_out={'output' in hooks['orig_k_ln_out']}, hook_in={'output' in hooks['hook_k_ln_in']}, hook_out={'output' in hooks['hook_k_ln_out']}")


entered qk_layernorm
entered esm3_use_org_rotary
entered esm3_use_torch_attention_calc
is sequence_id None? False
esm3 or esmc bias: None
entered qk_layernorm
entered esm3_use_org_rotary
entered esm3_use_torch_attention_calc
is sequence_id None? False
esm3 or esmc bias: None
entered qk_layernorm
entered esm3_use_org_rotary
entered esm3_use_torch_attention_calc
is sequence_id None? False
esm3 or esmc bias: None
entered qk_layernorm
entered esm3_use_org_rotary
entered esm3_use_torch_attention_calc
is sequence_id None? False
esm3 or esmc bias: None
entered qk_layernorm
entered esm3_use_org_rotary
entered esm3_use_torch_attention_calc
is sequence_id None? False
esm3 or esmc bias: None
entered qk_layernorm
entered esm3_use_org_rotary
entered esm3_use_torch_attention_calc
is sequence_id None? False
esm3 or esmc bias: None
entered qk_layernorm
entered esm3_use_org_rotary
entered esm3_use_torch_attention_calc
is sequence_id None? False
esm3 or esmc bias: None
entered qk_layernorm
entered esm3_

In [17]:
# Helper function to compare tensors
def compare(name, orig_key, hook_key, rtol=1e-6, atol=1e-6):
    o = hooks[orig_key]['output']
    h = hooks[hook_key]['output']
    if o.dtype != h.dtype: h = h.to(o.dtype)
    diff = torch.abs(o - h)
    max_diff = torch.max(diff).item()
    identical = torch.allclose(o, h, rtol=rtol, atol=atol)
    print(f"{name}: max_diff={max_diff:.8f}, identical={identical}")
    if max_diff > 1e-6:
        pos = torch.unravel_index(torch.argmax(diff), diff.shape)
        print(f"  Max diff at {pos}: orig={o[pos]:.6f}, hook={h[pos]:.6f}, diff={diff[pos]:.6f}")

# Compare all components
compare("LN_QKV vs LN1", 'orig_ln_qkv', 'hook_ln1')
compare("q_ln IN", 'orig_q_ln_in', 'hook_q_ln_in')
compare("q_ln OUT", 'orig_q_ln_out', 'hook_q_ln_out')
compare("k_ln IN", 'orig_k_ln_in', 'hook_k_ln_in')
compare("k_ln OUT", 'orig_k_ln_out', 'hook_k_ln_out')
compare("Rotary Q IN", 'orig_rotary_q_in', 'hook_rotary_q_in')
compare("Rotary K IN", 'orig_rotary_k_in', 'hook_rotary_k_in')
compare("Rotary Q OUT", 'orig_rotary_q_out', 'hook_rotary_q_out')
compare("Rotary K OUT", 'orig_rotary_k_out', 'hook_rotary_k_out')
compare("Attention", 'orig_attn', 'hook_attn')


LN_QKV vs LN1: max_diff=0.00000000, identical=True
q_ln IN: max_diff=0.00000143, identical=True
  Max diff at (tensor(0, device='cuda:0'), tensor(3, device='cuda:0'), tensor(512, device='cuda:0')): orig=1.355702, hook=1.355703, diff=0.000001
q_ln OUT: max_diff=0.00001240, identical=False
  Max diff at (tensor(0, device='cuda:0'), tensor(3, device='cuda:0'), tensor(512, device='cuda:0')): orig=12.555263, hook=12.555275, diff=0.000012
k_ln IN: max_diff=0.00000238, identical=True
  Max diff at (tensor(0, device='cuda:0'), tensor(15, device='cuda:0'), tensor(688, device='cuda:0')): orig=2.167520, hook=2.167517, diff=0.000002
k_ln OUT: max_diff=0.00000620, identical=False
  Max diff at (tensor(0, device='cuda:0'), tensor(21, device='cuda:0'), tensor(928, device='cuda:0')): orig=-7.032105, hook=-7.032112, diff=0.000006
Rotary Q IN: max_diff=0.00001240, identical=False
  Max diff at (tensor(0, device='cuda:0'), tensor(3, device='cuda:0'), tensor(8, device='cuda:0'), tensor(0, device='cuda:0')

In [18]:
assert torch.equal(esmc_original.transformer.blocks[0].attn.q_ln.weight.data, esmc_hooked.blocks[0].attn.q_ln.weight.data)
assert esmc_original.transformer.blocks[0].attn.q_ln.bias is None and esmc_hooked.blocks[0].attn.q_ln.bias is None
assert torch.equal(esmc_original.transformer.blocks[0].attn.k_ln.weight.data, esmc_hooked.blocks[0].attn.k_ln.weight.data)
assert esmc_original.transformer.blocks[0].attn.k_ln.bias is None and esmc_hooked.blocks[0].attn.k_ln.bias is None


In [19]:
# Compare FFN/MLP outputs
of = hooks['orig_ffn']['output']
hm = hooks['hook_mlp']['output']
if of.dtype != hm.dtype: hm = hm.to(of.dtype)
diff = torch.abs(of - hm)
print(f"FFN/MLP: max_diff={torch.max(diff):.8f}, identical={torch.allclose(of, hm, rtol=1e-6, atol=1e-6)}")
if torch.max(diff) > 1e-6:
    pos = torch.unravel_index(torch.argmax(diff), diff.shape)
    print(f"  Max diff at {pos}: orig={of[pos]:.6f}, hook={hm[pos]:.6f}, diff={diff[pos]:.6f}")


FFN/MLP: max_diff=0.00000334, identical=True
  Max diff at (tensor(0, device='cuda:0'), tensor(29, device='cuda:0'), tensor(212, device='cuda:0')): orig=6.067447, hook=6.067444, diff=0.000003


In [20]:
# Final output comparison
ol = out1.sequence_logits
hl = out2
if ol.dtype != hl.dtype: hl = hl.to(ol.dtype)
diff = torch.abs(ol - hl)
print(f"Final: max_diff={torch.max(diff):.8f}, within_tol={torch.allclose(ol, hl, rtol=1.3e-6, atol=4e-5)}")

# Cleanup
for h in [h1, h2, h3, h4, h5, h6_orig, h7, h8, h9, h10, h11, h12]: h.remove()


Final: max_diff=0.00007534, within_tol=False


In [21]:
print(ol)

tensor([[[-40.1641, -40.1526, -40.1775,  ..., -40.1413, -40.1886, -40.1702],
         [-37.4839, -37.5041, -37.5147,  ..., -37.4731, -37.5277, -37.5132],
         [-36.0404, -36.0813, -36.0888,  ..., -36.1025, -36.1244, -36.0783],
         ...,
         [-37.7287, -37.7334, -37.7655,  ..., -37.7925, -37.7991, -37.7551],
         [-35.2574, -35.2673, -35.2886,  ..., -35.3117, -35.3367, -35.3013],
         [-34.4202, -34.4303, -34.4586,  ..., -34.4555, -34.4898, -34.4474]]],
       device='cuda:0')


In [22]:
print(hl)

tensor([[[-40.1641, -40.1526, -40.1775,  ..., -40.1413, -40.1886, -40.1702],
         [-37.4839, -37.5041, -37.5147,  ..., -37.4731, -37.5277, -37.5132],
         [-36.0404, -36.0813, -36.0888,  ..., -36.1025, -36.1244, -36.0783],
         ...,
         [-37.7288, -37.7334, -37.7655,  ..., -37.7925, -37.7991, -37.7551],
         [-35.2574, -35.2673, -35.2886,  ..., -35.3117, -35.3367, -35.3013],
         [-34.4202, -34.4303, -34.4586,  ..., -34.4555, -34.4898, -34.4474]]],
       device='cuda:0')
